# Question Understanding & Decomposition Pipeline

This notebook walks through the **Pilot 2 question-understanding workflow** step by step.
Each stage uses the configured LLM (Scayle `qwen3` by default) — no Break model needed.

| Step | Name | What it does |
|------|------|-------------|
| 1 | **Route** | Extract the target concept being compared |
| 2 | **Disambiguate** | Parse periods, languages → comparison slices |
| 3 | **Extend Knowledge** | LLM generates thematic facets, period-specific lemmas, comparative lenses |
| 4 | **Decompose** | LLM produces sub-questions with `#k` dependencies |

In [1]:
import sys, os, json, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import HTML, display, Markdown

from src.llm import configure_refinement_lm, llm_endpoint, selected_model
from src.routing.router import route_query
from src.schemas.slice import ComparisonSlice, ParsedQueryIntent, parse_query_intent
from src.knowledge_extension.expander import expand_knowledge, knowledge_extension_payload

# ── LLM provider (change to "ollama" or "openai" as needed) ──
LLM_CHOICE = None   # None → reads LLM_PROVIDER from .env (default: scayle)

label = configure_refinement_lm(prefer=LLM_CHOICE, force=True)
print(f"✓ LLM configured: {label}")
print(f"  endpoint: {llm_endpoint()}")

✓ LLM configured: scayle:qwen3
  endpoint: https://172.16.59.5:32443/api


## Test questions

Three benchmark questions plus one custom question to test concept extraction from the LLM.

In [2]:
QUESTIONS = [
    "How did descriptions of Dresden differ in English sources from 1800 and 1900?",
    "How was marriage described in German texts from the 1850s versus English texts from the 1850s?",
    "How was love portrayed in 1820 compared to 1920?",
    "How did the concept of freedom change in English newspapers from the 1830s versus the 1930s?",
]

for i, q in enumerate(QUESTIONS, 1):
    print(f"  Q{i}: {q}")

  Q1: How did descriptions of Dresden differ in English sources from 1800 and 1900?
  Q2: How was marriage described in German texts from the 1850s versus English texts from the 1850s?
  Q3: How was love portrayed in 1820 compared to 1920?
  Q4: How did the concept of freedom change in English newspapers from the 1830s versus the 1930s?


---
## Step 1 — Route

The router asks the LLM to extract the **concept being compared**. This is the first pass at understanding the question — what is the user asking about?

In [3]:
route_results = {}

for i, q in enumerate(QUESTIONS, 1):
    t0 = time.time()
    decision = route_query(q)
    elapsed = time.time() - t0
    route_results[q] = decision
    print(f"Q{i}  ({elapsed:.1f}s)")
    print(f"  domain  : {decision.domain}")
    print(f"  concept : {decision.concept or '—'}")
    print()

Q1  (3.8s)
  domain  : comparative
  concept : descriptions of Dresden in English sources



Q2  (1.5s)
  domain  : comparative
  concept : marriage descriptions in German and English texts from the 1850s



Q3  (0.9s)
  domain  : comparative
  concept : love



Q4  (0.8s)
  domain  : comparative
  concept : freedom



---
## Step 2 — Disambiguate

Parse the question into **comparison slices** — each slice is a (period, language) window into a corpus. The local regex parser handles explicit dates and language names. When regex misses the concept, we fall back to the LLM-extracted concept from Step 1.

In [4]:
disamb_results = {}

for i, q in enumerate(QUESTIONS, 1):
    llm_concept = route_results[q].concept or ""
    intent, errors = parse_query_intent(q)

    # If local regex missed the concept but the LLM found one, retry with a patched query
    if intent is None and llm_concept and any("concept" in e.lower() for e in errors):
        tail = q.split("in ", 1)[-1] if " in " in q else q
        patched = f"How were descriptions of {llm_concept} different in {tail}"
        intent, errors = parse_query_intent(patched)
        if intent:
            intent.original_query = q
            intent.target_concept = llm_concept

    disamb_results[q] = intent
    print(f"Q{i}: {q[:70]}...")

    if intent is None:
        print(f"  ✗ {errors}")
        print()
        continue

    print(f"  concept   : {intent.target_concept}")
    print(f"  dimension : {intent.dimension}")
    print(f"  slices:")
    for s in intent.slices:
        print(f"    {s.slice_id} │ {s.label:30s} │ {s.period_start}–{s.period_end} │ {s.language}")
    print()

Q1: How did descriptions of Dresden differ in English sources from 1800 an...
  concept   : Dresden
  dimension : temporal
  slices:
    slice_1 │ 1800                           │ 1800–1800 │ English
    slice_2 │ 1900                           │ 1900–1900 │ English

Q2: How was marriage described in German texts from the 1850s versus Engli...
  concept   : marriage
  dimension : multi-dimensional
  slices:
    slice_1 │ German sources, 1850s          │ 1850–1859 │ German
    slice_2 │ English sources, 1850s         │ 1850–1859 │ English

Q3: How was love portrayed in 1820 compared to 1920?...
  concept   : love
  dimension : temporal
  slices:
    slice_1 │ 1820                           │ 1820–1820 │ English
    slice_2 │ 1920                           │ 1920–1920 │ English

Q4: How did the concept of freedom change in English newspapers from the 1...
  concept   : freedom
  dimension : temporal
  slices:
    slice_1 │ 1830s                          │ 1830–1839 │ English
    slice_2 

---
## Step 3 — Extend Knowledge

For each successfully disambiguated question, the LLM expands the concept into:

- **Thematic facets** — topical dimensions of the concept
- **Slice lemmas** — period- and language-specific vocabulary (e.g. German terms for the German slice)
- **Feature lenses** — comparative metrics to measure across slices

In [5]:
extend_results = {}

for i, q in enumerate(QUESTIONS, 1):
    intent = disamb_results[q]
    if intent is None:
        print(f"Q{i}: skipped (no slices)")
        print()
        continue

    concept = intent.target_concept
    t0 = time.time()
    expansion = expand_knowledge(q, concept, intent.slices)
    elapsed = time.time() - t0
    payload = knowledge_extension_payload(expansion)
    extend_results[q] = payload

    print(f"Q{i}  ({elapsed:.1f}s)  concept = {concept}")
    print(f"  facets : {payload.get('thematic_facets', [])}")
    print(f"  lenses : {payload.get('feature_lenses', [])}")
    for s in intent.slices:
        lemmas = payload.get(s.slice_id, [])
        print(f"  {s.slice_id} ({s.label})")
        print(f"    lemmas: {lemmas}")
    print()

Q1  (4.7s)  concept = Dresden
  facets : ['cultural heritage', 'urban landscape', 'political symbolism', 'war and destruction', 'tourism and travel']
  lenses : ['frequency of cultural references', 'tone and sentiment', 'geopolitical framing', 'descriptive focus']
  slice_1 (1800)
    lemmas: ['Dresden', 'Saxon capital', 'Baroque city', 'artistic center', 'German city']
  slice_2 (1900)
    lemmas: ['Dresden', 'Saxon capital', 'modern city', 'cultural hub', 'German city']



Q2  (6.6s)  concept = marriage
  facets : ['marriage', 'wedding', 'spousal roles', 'marital obligations', 'courtship', 'domestic life', 'legal aspects of marriage', 'religious aspects of marriage']
  lenses : ['frequency of terms', 'co-occurring concepts', 'sentiment associated with terms', 'legal vs. emotional framing']
  slice_1 (German sources, 1850s)
    lemmas: ['Ehe', 'Hochzeit', 'Ehepartner', 'Ehevertrag', 'Trauung', 'Eheschließung', 'Ehewohnung', 'Ehebruch']
  slice_2 (English sources, 1850s)
    lemmas: ['marriage', 'wedding', 'spouse', 'marital contract', 'ceremony', 'marital union', 'domestic life', 'adultery']



Q3  (5.3s)  concept = love
  facets : ['romantic love', 'courtship', 'passion', 'marriage', 'emotional expression', 'social norms', 'literary conventions']
  lenses : ['frequency of emotional terms', 'context of romantic relationships', 'literary tropes', 'social expectations']
  slice_1 (1820)
    lemmas: ['love', 'adoration', 'devotion', 'fondness', 'courtship', 'passion', 'tenderness']
  slice_2 (1920)
    lemmas: ['love', 'romance', 'affection', 'desire', 'relationship', 'emotions', 'attraction']



Q4  (6.1s)  concept = freedom
  facets : ['freedom', 'liberty', 'rights', 'autonomy', 'independence', 'emancipation', 'self-determination', 'political freedom', 'civil liberties', 'individualism']
  lenses : ['frequency of terms', 'collocational patterns', 'sentiment analysis', 'contextual usage']
  slice_1 (1830s)
    lemmas: ['freedom', 'liberty', 'rights', 'emancipation', 'independence', 'self-government', 'free speech', 'free press']
  slice_2 (1930s)
    lemmas: ['freedom', 'liberty', 'rights', 'autonomy', 'independence', 'self-determination', 'civil liberties', 'democracy', 'individualism', 'freedom of expression']



---
## Step 4 — Decompose (knowledge-informed, few-shot)

The extended knowledge from Step 3 is now **input to the decomposition**: thematic facets tell the LLM *what dimensions to probe*, slice lemmas tell it *what vocabulary to expect*, and feature lenses tell it *what to compare*.

This is the DSP pipeline: **Demonstrate** (benchmark few-shots) → **Search/Extend** (facets, lemmas, lenses) → **Predict** (sub-questions informed by the enriched context).

In [6]:
import dspy
from src.decomposition.benchmark_fewshots import iter_decompositions

# ── Signature: knowledge-informed decomposition ──
class DecomposeQuestion(dspy.Signature):
    """Decompose a comparative historical question into sub-questions.

    You are given:
    - the question and the concept being compared
    - the comparison slices (time periods or languages)
    - thematic facets: topical dimensions discovered for this concept
    - slice-specific lemmas: period- and language-accurate vocabulary for each slice
    - feature lenses: the comparative metrics to measure across slices

    Use the facets and lemmas to craft sub-questions that probe specific
    dimensions of the concept in each slice. Use the lenses to shape the
    final synthesis question. Write full, self-contained questions.
    """
    question: str = dspy.InputField()
    concept: str = dspy.InputField(desc="The concept being compared")
    slices: str = dspy.InputField(desc="The comparison slices (periods / languages)")
    thematic_facets: str = dspy.InputField(desc="Topical dimensions of the concept")
    slice_lemmas: str = dspy.InputField(desc="Period- and language-specific vocabulary per slice")
    feature_lenses: str = dspy.InputField(desc="Comparative metrics to measure across slices")
    sub_questions: list[str] = dspy.OutputField(
        desc="Ordered list of sub-questions. Last one is the synthesis question."
    )

# ── Build few-shot demos from benchmark gold ──
def build_demos(exclude_question: str = "", limit: int = 3) -> list[dspy.Example]:
    demos = []
    for item in iter_decompositions():
        if item["question"].strip() == exclude_question.strip():
            continue
        subs = list(item["subquestions"])
        refs = " and ".join(f"sub-question {j}" for j in range(1, len(subs) + 1))
        subs.append(f"How do these accounts differ across the sources, considering {refs}?")
        slice_labels = [s.label for s in item["slices"]]
        demos.append(
            dspy.Example(
                question=item["question"],
                concept=item["concept"],
                slices=", ".join(slice_labels),
                thematic_facets=item["concept"],          # simple for demos
                slice_lemmas=", ".join(slice_labels),      # simple for demos
                feature_lenses="frequency, tone, framing",  # simple for demos
                sub_questions=subs,
            ).with_inputs("question", "concept", "slices", "thematic_facets", "slice_lemmas", "feature_lenses")
        )
    return demos[:limit]

# ── Run decomposition with knowledge context ──
predictor = dspy.Predict(DecomposeQuestion)
decompose_results = {}

for i, q in enumerate(QUESTIONS, 1):
    intent = disamb_results[q]
    if intent is None:
        print(f"Q{i}: skipped (no slices)")
        print()
        continue

    concept = intent.target_concept
    slice_labels = ", ".join(s.label for s in intent.slices)

    # Get extended knowledge from Step 3
    ext = extend_results.get(q, {})
    facets_str = ", ".join(ext.get("thematic_facets", []))
    lenses_str = ", ".join(ext.get("feature_lenses", []))
    lemma_parts = []
    for s in intent.slices:
        lemmas = ext.get(s.slice_id, [])
        if lemmas:
            lemma_parts.append(f"{s.label}: {', '.join(lemmas)}")
    lemmas_str = "; ".join(lemma_parts) if lemma_parts else concept

    # Set few-shot demos (hold out the current question)
    predictor.demos = build_demos(exclude_question=q, limit=3)

    t0 = time.time()
    try:
        result = predictor(
            question=q,
            concept=concept,
            slices=slice_labels,
            thematic_facets=facets_str,
            slice_lemmas=lemmas_str,
            feature_lenses=lenses_str,
        )
        subs = result.sub_questions
        if isinstance(subs, str):
            subs = [s.strip() for s in subs.split("\n") if s.strip()]
    except Exception as exc:
        subs = []
        print(f"Q{i}: error — {exc}")
    elapsed = time.time() - t0

    decompose_results[q] = subs
    print(f"Q{i}  ({elapsed:.1f}s)  →  {len(subs)} sub-questions")
    print(f"  question : {q}")
    print(f"  concept  : {concept}")
    print(f"  slices   : {slice_labels}")
    print(f"  facets   : {facets_str}")
    print(f"  lenses   : {lenses_str}")
    print(f"  lemmas   : {lemmas_str}")
    print()
    for j, sq in enumerate(subs, 1):
        print(f"  {j}. {sq}")
    print()

Q1  (6.4s)  →  6 sub-questions
  question : How did descriptions of Dresden differ in English sources from 1800 and 1900?
  concept  : Dresden
  slices   : 1800, 1900
  facets   : cultural heritage, urban landscape, political symbolism, war and destruction, tourism and travel
  lenses   : frequency of cultural references, tone and sentiment, geopolitical framing, descriptive focus
  lemmas   : 1800: Dresden, Saxon capital, Baroque city, artistic center, German city; 1900: Dresden, Saxon capital, modern city, cultural hub, German city

  1. How is Dresden described in terms of its cultural heritage in 1800 and 1900 English sources?
  2. What aspects of Dresden’s urban landscape are emphasized in 1800 and 1900 English sources?
  3. How is Dresden framed politically or symbolically in 1800 and 1900 English sources?
  4. What references to war, destruction, or rebuilding are present in 1800 and 1900 English sources?
  5. How is Dresden portrayed in the context of tourism and travel in 1800

Q2  (8.9s)  →  8 sub-questions
  question : How was marriage described in German texts from the 1850s versus English texts from the 1850s?
  concept  : marriage
  slices   : German sources, 1850s, English sources, 1850s
  facets   : marriage, wedding, spousal roles, marital obligations, courtship, domestic life, legal aspects of marriage, religious aspects of marriage
  lenses   : frequency of terms, co-occurring concepts, sentiment associated with terms, legal vs. emotional framing
  lemmas   : German sources, 1850s: Ehe, Hochzeit, Ehepartner, Ehevertrag, Trauung, Eheschließung, Ehewohnung, Ehebruch; English sources, 1850s: marriage, wedding, spouse, marital contract, ceremony, marital union, domestic life, adultery

  1. How is the concept of marriage defined in German sources from the 1850s, using terms like Ehe, Hochzeit, and Eheschließung?
  2. What are the key terms and phrases used to describe marriage in English sources from the 1850s, such as marriage, wedding, and marital uni

Q3  (9.2s)  →  6 sub-questions
  question : How was love portrayed in 1820 compared to 1920?
  concept  : love
  slices   : 1820, 1920
  facets   : romantic love, courtship, passion, marriage, emotional expression, social norms, literary conventions
  lenses   : frequency of emotional terms, context of romantic relationships, literary tropes, social expectations
  lemmas   : 1820: love, adoration, devotion, fondness, courtship, passion, tenderness; 1920: love, romance, affection, desire, relationship, emotions, attraction

  1. How is the concept of love described in the 1820 context, particularly in relation to romantic love, courtship, and social norms?
  2. What terms and expressions are used to convey emotional states like passion, tenderness, or devotion in 1820?
  3. How is the portrayal of love in 1920 different, especially in terms of emotional expression, relationships, and literary conventions?
  4. What terms and expressions are used to convey emotional states like romance, 

Q4  (8.6s)  →  5 sub-questions
  question : How did the concept of freedom change in English newspapers from the 1830s versus the 1930s?
  concept  : freedom
  slices   : 1830s, 1930s
  facets   : freedom, liberty, rights, autonomy, independence, emancipation, self-determination, political freedom, civil liberties, individualism
  lenses   : frequency of terms, collocational patterns, sentiment analysis, contextual usage
  lemmas   : 1830s: freedom, liberty, rights, emancipation, independence, self-government, free speech, free press; 1930s: freedom, liberty, rights, autonomy, independence, self-determination, civil liberties, democracy, individualism, freedom of expression

  1. What are the most frequent terms used to express the concept of freedom in the 1830s and the 1930s, respectively?
  2. What are the common collocational patterns associated with 'freedom' in the 1830s and the 1930s, and how do they differ?
  3. What is the sentiment or tone associated with the term 'freedom' i

---
## Step 4b — ChainOfThought decomposition

`dspy.ChainOfThought` adds a hidden **reasoning** field before the output. The LLM reasons through the facets, lemmas and lenses before producing sub-questions — giving us a visible trace of *why* each sub-question was chosen.

In [7]:
# ── ChainOfThought: reason, then decompose ──
cot_predictor = dspy.ChainOfThought(DecomposeQuestion)
cot_results = {}

for i, q in enumerate(QUESTIONS, 1):
    intent = disamb_results[q]
    if intent is None:
        print(f"Q{i}: skipped")
        continue

    concept = intent.target_concept
    slice_labels = ", ".join(s.label for s in intent.slices)
    ext = extend_results.get(q, {})
    facets_str = ", ".join(ext.get("thematic_facets", []))
    lenses_str = ", ".join(ext.get("feature_lenses", []))
    lemma_parts = []
    for s in intent.slices:
        lemmas = ext.get(s.slice_id, [])
        if lemmas:
            lemma_parts.append(f"{s.label}: {', '.join(lemmas)}")
    lemmas_str = "; ".join(lemma_parts) if lemma_parts else concept

    cot_predictor.demos = build_demos(exclude_question=q, limit=3)

    t0 = time.time()
    try:
        result = cot_predictor(
            question=q, concept=concept, slices=slice_labels,
            thematic_facets=facets_str, slice_lemmas=lemmas_str, feature_lenses=lenses_str,
        )
        subs = result.sub_questions
        if isinstance(subs, str):
            subs = [s.strip() for s in subs.split("\n") if s.strip()]
        reasoning = getattr(result, "reasoning", "")
    except Exception as exc:
        subs, reasoning = [], str(exc)
    elapsed = time.time() - t0

    cot_results[q] = {"subs": subs, "reasoning": reasoning}
    print(f"Q{i}  ({elapsed:.1f}s)  →  {len(subs)} sub-questions")
    print(f"  REASONING: {reasoning[:300]}...")
    print()
    for j, sq in enumerate(subs, 1):
        print(f"  {j}. {sq}")
    print()

Q1  (11.4s)  →  7 sub-questions
  REASONING: To address the comparative question about how descriptions of Dresden differ in English sources from 1800 and 1900, we need to examine the concept of Dresden through its thematic facets (cultural heritage, urban landscape, political symbolism, war and destruction, tourism and travel) and the specifi...

  1. How was Dresden's cultural heritage described in English sources from 1800, and what specific terms or references were used?
  2. How was Dresden's urban landscape portrayed in English sources from 1800, and what descriptive language was employed?
  3. What political symbolism was associated with Dresden in English sources from 1800, and how was it framed geographically or nationally?
  4. How was Dresden's cultural heritage described in English sources from 1900, and what specific terms or references were used?
  5. How was Dresden's urban landscape portrayed in English sources from 1900, and what descriptive language was employed?
  6.

Q2  (15.3s)  →  9 sub-questions
  REASONING: The question asks for a comparison of how marriage was described in German and English texts from the 1850s. The thematic facets provide a range of topics related to marriage, such as spousal roles, legal aspects, and religious aspects. The slice lemmas offer period- and language-specific vocabulary...

  1. What terms related to marriage (e.g., Ehe, Hochzeit, marriage, wedding) were most frequently used in German texts from the 1850s?
  2. What terms related to marriage (e.g., Ehe, Hochzeit, marriage, wedding) were most frequently used in English texts from the 1850s?
  3. What concepts co-occurred with marriage-related terms in German texts from the 1850s (e.g., spousal roles, legal aspects)?
  4. What concepts co-occurred with marriage-related terms in English texts from the 1850s (e.g., spousal roles, legal aspects)?
  5. What sentiment was associated with marriage-related terms in German texts from the 1850s?
  6. What sentiment was ass

Q3  (10.6s)  →  5 sub-questions
  REASONING: To address the question of how love was portrayed in 1820 compared to 1920, we need to examine the concept of love through its thematic dimensions (romantic love, courtship, passion, etc.) and the specific vocabulary used in each time period. The sub-questions will explore how these dimensions were ...

  1. How was romantic love expressed in 1820, and what role did terms like 'adoration,' 'devotion,' and 'tenderness' play in its portrayal?
  2. What were the social norms and literary conventions surrounding courtship in 1820, and how did they shape the portrayal of love?
  3. How was love portrayed in 1920, and what role did terms like 'romance,' 'desire,' and 'attraction' play in its expression?
  4. What were the social norms and literary conventions surrounding relationships in 1920, and how did they shape the portrayal of love?
  5. How do the portrayals of love in 1820 and 1920 compare in terms of the frequency of emotional terms, the 

Q4  (15.4s)  →  7 sub-questions
  REASONING: To address the question of how the concept of freedom changed in English newspapers from the 1830s to the 1930s, we need to examine the thematic dimensions of freedom (e.g., liberty, rights, autonomy) in each time period. The slice-specific lemmas provide the vocabulary relevant to each decade, and ...

  1. What is the frequency of the terms 'freedom', 'liberty', 'rights', 'emancipation', 'independence', and 'self-government' in English newspapers from the 1830s?
  2. What is the frequency of the terms 'freedom', 'liberty', 'rights', 'autonomy', 'independence', 'self-determination', 'civil liberties', 'democracy', 'individualism', and 'freedom of expression' in English newspapers from the 1930s?
  3. What are the collocational patterns of the term 'freedom' in the 1830s, and how do they relate to the thematic facets of liberty, rights, emancipation, and independence?
  4. What are the collocational patterns of the term 'freedom' in the 1930

---
## Step 4c — Refine with quality feedback

`dspy.Refine` runs the module up to N times. After each attempt, it scores the result with a reward function and auto-generates feedback to improve the next attempt. This gives the LLM a chance to self-correct.

**Reward function** checks:
- Does each slice get at least one dedicated sub-question?
- Are facets or lemmas referenced in the sub-questions?
- Is there a synthesis question at the end?
- Penalizes too few or too many sub-questions.

In [8]:
# ── Reward function for decomposition quality ──
def decompose_reward(inputs: dict, pred) -> float:
    """Score a decomposition: 0.0 (bad) to 1.0 (excellent)."""
    subs = pred.sub_questions
    if isinstance(subs, str):
        subs = [s.strip() for s in subs.split("\n") if s.strip()]
    if not subs:
        return 0.0

    score = 0.0
    slices_text = inputs.get("slices", "")
    facets_text = inputs.get("thematic_facets", "").lower()
    lemmas_text = inputs.get("slice_lemmas", "").lower()
    lenses_text = inputs.get("feature_lenses", "").lower()
    joined = " ".join(subs).lower()

    # 1. Each slice should appear in at least one sub-question (0.3)
    slice_labels = [s.strip().lower() for s in slices_text.split(",")]
    slice_hits = sum(1 for sl in slice_labels if sl and sl in joined)
    score += 0.3 * (slice_hits / max(len(slice_labels), 1))

    # 2. Facets or lemmas appear in sub-questions (0.3)
    facet_words = [f.strip() for f in facets_text.split(",") if f.strip()]
    lemma_words = [w.strip() for w in lemmas_text.replace(";", ",").split(",") if w.strip() and ":" not in w]
    knowledge_words = facet_words + lemma_words
    if knowledge_words:
        hits = sum(1 for w in knowledge_words if w in joined)
        score += 0.3 * min(hits / max(len(knowledge_words) * 0.3, 1), 1.0)

    # 3. Synthesis question at the end (0.2)
    last = subs[-1].lower() if subs else ""
    synthesis_words = ["differ", "compare", "change", "shift", "contrast", "how do"]
    if any(w in last for w in synthesis_words):
        score += 0.2

    # 4. Reasonable count: 3–8 sub-questions (0.2)
    if 3 <= len(subs) <= 8:
        score += 0.2
    elif len(subs) == 2:
        score += 0.1

    return round(min(score, 1.0), 3)


# ── Refine module: CoT + retry with feedback ──
cot_module = dspy.ChainOfThought(DecomposeQuestion)
refine_module = dspy.Refine(
    module=cot_module,
    N=3,
    reward_fn=decompose_reward,
    threshold=0.85,
)
refine_results = {}

for i, q in enumerate(QUESTIONS, 1):
    intent = disamb_results[q]
    if intent is None:
        print(f"Q{i}: skipped")
        continue

    concept = intent.target_concept
    slice_labels = ", ".join(s.label for s in intent.slices)
    ext = extend_results.get(q, {})
    facets_str = ", ".join(ext.get("thematic_facets", []))
    lenses_str = ", ".join(ext.get("feature_lenses", []))
    lemma_parts = []
    for s in intent.slices:
        lemmas = ext.get(s.slice_id, [])
        if lemmas:
            lemma_parts.append(f"{s.label}: {', '.join(lemmas)}")
    lemmas_str = "; ".join(lemma_parts) if lemma_parts else concept

    cot_module.demos = build_demos(exclude_question=q, limit=3)

    t0 = time.time()
    try:
        result = refine_module(
            question=q, concept=concept, slices=slice_labels,
            thematic_facets=facets_str, slice_lemmas=lemmas_str, feature_lenses=lenses_str,
        )
        subs = result.sub_questions
        if isinstance(subs, str):
            subs = [s.strip() for s in subs.split("\n") if s.strip()]
    except Exception as exc:
        subs = []
        print(f"Q{i}: error — {exc}")
    elapsed = time.time() - t0

    # Score the final result
    reward = decompose_reward(
        {"slices": slice_labels, "thematic_facets": facets_str,
         "slice_lemmas": lemmas_str, "feature_lenses": lenses_str},
        result if subs else type("P", (), {"sub_questions": []})(),
    )
    refine_results[q] = {"subs": subs, "reward": reward}
    print(f"Q{i}  ({elapsed:.1f}s)  reward={reward:.3f}  →  {len(subs)} sub-questions")
    for j, sq in enumerate(subs, 1):
        print(f"  {j}. {sq}")
    print()

Q1  (11.9s)  reward=0.931  →  7 sub-questions
  1. How was Dresden's cultural heritage described in English sources from 1800, and what specific terms or references were used?
  2. How was Dresden's urban landscape portrayed in English sources from 1800, and what descriptive language was employed?
  3. What political symbolism was associated with Dresden in English sources from 1800, and how was it framed geographically or nationally?
  4. How was Dresden's cultural heritage described in English sources from 1900, and what specific terms or references were used?
  5. How was Dresden's urban landscape portrayed in English sources from 1900, and what descriptive language was employed?
  6. What political symbolism was associated with Dresden in English sources from 1900, and how was it framed geographically or nationally?
  7. How did the frequency of cultural references, tone and sentiment, geopolitical framing, and descriptive focus differ in English descriptions of Dresden between 180

2026/09/28 14:31:47 WARNING dspy.predict.predict: Type mismatch for field 'target_threshold': expected float based on given Signature, but the provided value is incompatible: 0.85.


2026/09/28 14:31:47 WARNING dspy.predict.predict: Type mismatch for field 'reward_value': expected float based on given Signature, but the provided value is incompatible: 0.6.


2026/09/28 14:31:47 WARNING dspy.predict.predict: Type mismatch for field 'module_names': expected list[str] based on given Signature, but the provided value is incompatible: [
  "predict"
].


Q2  (50.8s)  reward=0.850  →  8 sub-questions
  1. What is the frequency of the term 'Ehe' in German texts from the 1850s compared to the term 'marriage' in English texts from the 1850s?
  2. How do the co-occurring concepts of 'Hochzeit' in German texts from the 1850s compare to the co-occurring concepts of 'wedding' in English texts from the 1850s?
  3. What is the sentiment associated with the term 'Ehebruch' in German texts from the 1850s compared to the sentiment associated with 'adultery' in English texts from the 1850s?
  4. How are the legal aspects of marriage, such as 'Ehevertrag' in German texts and 'marital contract' in English texts, described in the 1850s?
  5. What are the religious aspects of marriage, such as 'Trauung' in German texts and 'ceremony' in English texts, emphasized in the 1850s?
  6. How are spousal roles and domestic life, such as 'Ehewohnung' in German texts and 'domestic life' in English texts, portrayed in the 1850s?
  7. How do the descriptions of cou

Q3  (10.4s)  reward=1.000  →  5 sub-questions
  1. How was romantic love expressed in 1820, and what role did terms like 'adoration,' 'devotion,' and 'tenderness' play in its portrayal?
  2. What were the social norms and literary conventions surrounding courtship in 1820, and how did they shape the portrayal of love?
  3. How was love portrayed in 1920, and what role did terms like 'romance,' 'desire,' and 'attraction' play in its expression?
  4. What were the social norms and literary conventions surrounding relationships in 1920, and how did they shape the portrayal of love?
  5. How do the portrayals of love in 1820 and 1920 compare in terms of the frequency of emotional terms, the context of romantic relationships, literary tropes, and social expectations?



Q4  (15.4s)  reward=1.000  →  7 sub-questions
  1. What is the frequency of the terms 'freedom', 'liberty', 'rights', 'emancipation', 'independence', and 'self-government' in English newspapers from the 1830s?
  2. What is the frequency of the terms 'freedom', 'liberty', 'rights', 'autonomy', 'independence', 'self-determination', 'civil liberties', 'democracy', 'individualism', and 'freedom of expression' in English newspapers from the 1930s?
  3. What are the collocational patterns of the term 'freedom' in the 1830s, and how do they relate to the thematic facets of liberty, rights, emancipation, and independence?
  4. What are the collocational patterns of the term 'freedom' in the 1930s, and how do they relate to the thematic facets of autonomy, self-determination, civil liberties, and democracy?
  5. What is the sentiment associated with the term 'freedom' in the 1830s, and how does it vary across the thematic facets of liberty, rights, and emancipation?
  6. What is the sentiment a

---
## Comparison: Predict vs ChainOfThought vs Refine

Side-by-side scores and sub-question counts for each DSPy strategy.

In [9]:
# Score the Predict results too, for fair comparison
predict_scored = {}
for q in QUESTIONS:
    intent = disamb_results[q]
    if intent is None:
        continue
    ext = extend_results.get(q, {})
    slice_labels = ", ".join(s.label for s in intent.slices)
    facets_str = ", ".join(ext.get("thematic_facets", []))
    lenses_str = ", ".join(ext.get("feature_lenses", []))
    lemma_parts = []
    for s in intent.slices:
        lemmas = ext.get(s.slice_id, [])
        if lemmas:
            lemma_parts.append(f"{s.label}: {', '.join(lemmas)}")
    lemmas_str = "; ".join(lemma_parts)

    subs = decompose_results.get(q, [])
    reward = decompose_reward(
        {"slices": slice_labels, "thematic_facets": facets_str,
         "slice_lemmas": lemmas_str, "feature_lenses": lenses_str},
        type("P", (), {"sub_questions": subs})(),
    )
    predict_scored[q] = reward

# Also score the CoT results
cot_scored = {}
for q in QUESTIONS:
    intent = disamb_results[q]
    if intent is None:
        continue
    ext = extend_results.get(q, {})
    slice_labels = ", ".join(s.label for s in intent.slices)
    facets_str = ", ".join(ext.get("thematic_facets", []))
    lenses_str = ", ".join(ext.get("feature_lenses", []))
    lemma_parts = []
    for s in intent.slices:
        lemmas = ext.get(s.slice_id, [])
        if lemmas:
            lemma_parts.append(f"{s.label}: {', '.join(lemmas)}")
    lemmas_str = "; ".join(lemma_parts)

    subs = cot_results.get(q, {}).get("subs", [])
    reward = decompose_reward(
        {"slices": slice_labels, "thematic_facets": facets_str,
         "slice_lemmas": lemmas_str, "feature_lenses": lenses_str},
        type("P", (), {"sub_questions": subs})(),
    )
    cot_scored[q] = reward

# Build comparison table
rows = []
for i, q in enumerate(QUESTIONS, 1):
    p_subs = decompose_results.get(q, [])
    c_subs = cot_results.get(q, {}).get("subs", [])
    r_data = refine_results.get(q, {})
    r_subs = r_data.get("subs", [])

    p_score = predict_scored.get(q, 0)
    c_score = cot_scored.get(q, 0)
    r_score = r_data.get("reward", 0)

    # Highlight best score
    best = max(p_score, c_score, r_score)
    def fmt(score, n):
        bold = " font-weight:bold;color:#0f6e56" if score == best and score > 0 else ""
        return f"<td style='padding:6px 10px;text-align:center;{bold}'>{score:.3f} ({n})</td>"

    rows.append(
        f"<tr><td style='padding:6px 10px'>Q{i}</td>"
        f"<td style='padding:6px 10px'>{q[:50]}...</td>"
        f"{fmt(p_score, len(p_subs))}"
        f"{fmt(c_score, len(c_subs))}"
        f"{fmt(r_score, len(r_subs))}</tr>"
    )

html = (
    '<table style="border-collapse:collapse;margin:10px 0;font-size:14px">'
    '<tr style="background:#f0f0f0">'
    "<th style='padding:6px 10px'>Q</th>"
    "<th style='padding:6px 10px'>Question</th>"
    "<th style='padding:6px 10px'>Predict<br><small>score (n)</small></th>"
    "<th style='padding:6px 10px'>ChainOfThought<br><small>score (n)</small></th>"
    "<th style='padding:6px 10px'>Refine<br><small>score (n)</small></th></tr>"
    + "\n".join(rows)
    + "</table>"
)
display(HTML(html))

Q,Question,Predictscore (n),ChainOfThoughtscore (n),Refinescore (n)
Q1,How did descriptions of Dresden differ in English ...,0.931 (6),0.931 (7),0.931 (7)
Q2,How was marriage described in German texts from th...,1.000 (8),0.600 (9),0.850 (8)
Q3,How was love portrayed in 1820 compared to 1920?...,1.000 (6),1.000 (5),1.000 (5)
Q4,How did the concept of freedom change in English n...,0.738 (5),1.000 (7),1.000 (7)


---
## Step 5 — Retrieve (Cross-Dataset Discovery smoke test)

Now that we have sub-questions, we connect to the **Cross-Dataset Discovery** service to retrieve relevant passages from the indexed corpora.

| Parameter | Value |
|-----------|-------|
| **API** | `POST /search/` on `https://datagems-dev.scayle.es/cross-dataset-discovery/` |
| **Auth** | Keycloak password grant via `swagger-client` + scope `openid cross-dataset-discovery-api offline_access` |
| **Search modes** | `sparse` (BM25), `dense` (semantic), `hybrid` (RRF) |
| **Existing client** | `CrossDatasetDiscoveryClient` in `src/orchestration/slice_executor.py` |
| **Query builder** | `build_search_query(concept, slice_label, facets, lemmas)` |

The pipeline builds **one search query per slice**, combining the concept, slice label, and period-specific lemmas from Step 3.


In [10]:
# ── Cross-Dataset Discovery: configuration & health check ──
import urllib.request, urllib.error, urllib.parse, json as _json, base64

from src.orchestration.slice_executor import (
    CrossDatasetDiscoveryClient,
    build_search_query,
)

DISCOVERY_URL = os.getenv(
    "CROSS_DATASET_DISCOVERY_URL",
    "https://datagems-dev.scayle.es/cross-dataset-discovery",
)
DISCOVERY_TOKEN = os.getenv("CROSS_DATASET_DISCOVERY_TOKEN", "")

print(f"Discovery URL : {DISCOVERY_URL}")
print(f"Token present : {'yes (' + DISCOVERY_TOKEN[:12] + '...)' if DISCOVERY_TOKEN else 'not set yet'}")

# ── Health check ──
health_url = f"{DISCOVERY_URL.rstrip('/')}/health"
print(f"\nHealth check  : GET {health_url}")
try:
    req = urllib.request.Request(health_url, headers={"Accept": "application/json"}, method="GET")
    with urllib.request.urlopen(req, timeout=15) as resp:
        health = _json.loads(resp.read().decode("utf-8"))
        print(f"  status  : {health.get('status', '?')}")
        print(f"  message : {health.get('message', '?')}")
        print("  ✓ Service is reachable")
except urllib.error.HTTPError as exc:
    print(f"  ✗ HTTP {exc.code}: {exc.read().decode('utf-8', errors='replace')[:200]}")
except urllib.error.URLError as exc:
    print(f"  ✗ Unreachable: {exc.reason}")
except Exception as exc:
    print(f"  ✗ Error: {exc}")

# ── Keycloak OIDC token ──
# Working template (same as curl):
#   client_id = swagger-client
#   scope     = openid cross-dataset-discovery-api offline_access
DG_USERNAME = os.getenv("DG_USERNAME", "dg-user-1")
DG_PASSWORD = os.getenv("DG_PASSWORD", "dg-user-1")
OIDC_CLIENT_ID = os.getenv("OIDC_CLIENT_ID", "swagger-client")
OIDC_SCOPE = os.getenv(
    "OIDC_SCOPE",
    "openid cross-dataset-discovery-api offline_access",
)

if not DISCOVERY_TOKEN:
    print("\n── Acquiring token via Keycloak OIDC ──")
    print(f"  client_id : {OIDC_CLIENT_ID}")
    print(f"  scope     : {OIDC_SCOPE}")
    print(f"  username  : {DG_USERNAME}")
    keycloak_token_url = (
        "https://datagems-dev.scayle.es/oauth/realms/dev"
        "/protocol/openid-connect/token"
    )
    form_body = urllib.parse.urlencode({
        "grant_type": "password",
        "client_id": OIDC_CLIENT_ID,
        "username": DG_USERNAME,
        "password": DG_PASSWORD,
        "scope": OIDC_SCOPE,
    }).encode("utf-8")
    try:
        req = urllib.request.Request(keycloak_token_url, data=form_body, headers={
            "Content-Type": "application/x-www-form-urlencoded",
            "Accept": "application/json",
        })
        with urllib.request.urlopen(req, timeout=15) as resp:
            token_data = _json.loads(resp.read().decode("utf-8"))
            DISCOVERY_TOKEN = token_data.get("access_token", "")
            print(f"  ✓ Token acquired (expires_in={token_data.get('expires_in')}s)")
            payload_b64 = DISCOVERY_TOKEN.split(".")[1]
            payload_b64 += "=" * (4 - len(payload_b64) % 4)
            claims = _json.loads(base64.urlsafe_b64decode(payload_b64))
            roles = claims.get("realm_access", {}).get("roles", [])
            print(f"  roles   : {roles}")
            needed = {"user", "dg_user", "dg_admin", "dg_system", "dg_dataset-curator"}
            if needed & set(roles):
                print("  ✓ Has required role for /search/")
            else:
                print(f"  ⚠ No matching realm role in JWT (search may still work via client scope)")
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")[:200]
        print(f"  ✗ Keycloak HTTP {exc.code}: {detail}")
    except Exception as exc:
        print(f"  ✗ {exc}")

# ── OpenAPI: available endpoints ──
print("\n── API endpoints (from OpenAPI spec) ──")
try:
    spec_url = f"{DISCOVERY_URL.rstrip('/')}/openapi.json"
    req = urllib.request.Request(spec_url, headers={"Accept": "application/json"})
    with urllib.request.urlopen(req, timeout=10) as resp:
        spec = _json.loads(resp.read().decode("utf-8"))
        for path, methods in spec.get("paths", {}).items():
            for method in methods:
                summary = methods[method].get("summary", "")
                secured = "🔒" if "security" in methods[method] else "🔓"
                print(f"  {secured} {method.upper():6s} {path:40s} {summary}")
except Exception as exc:
    print(f"  Could not fetch OpenAPI spec: {exc}")

Discovery URL : https://datagems-dev.scayle.es/cross-dataset-discovery
Token present : not set yet

Health check  : GET https://datagems-dev.scayle.es/cross-dataset-discovery/health


  status  : ok
  message : All dependencies are healthy.
  ✓ Service is reachable

── Acquiring token via Keycloak OIDC ──
  client_id : swagger-client
  scope     : openid cross-dataset-discovery-api offline_access
  username  : dg-user-1


  ✓ Token acquired (expires_in=300s)
  roles   : ['dg_dataset-curator', 'default-roles-dev', 'dg_dataset-uploader', 'dg_user', 'dg_col-manage', 'offline_access', 'uma_authorization']
  ✓ Has required role for /search/

── API endpoints (from OpenAPI spec) ──
  🔓 GET    /                                        Read Root
  🔒 POST   /search/                                 Perform Search
  🔒 POST   /search-ap/                              Perform Search From Ap
  🔒 POST   /corpus-analysis-search/                 Corpus Search
  🔒 POST   /ingest/                                 Ingest Dataset
  🔒 GET    /ingest/{job_id}                         Get Ingestion Status
  🔒 DELETE /dataset/{dataset_id}                    Delete Dataset
  🔓 GET    /health                                  Health Check


### Example: hybrid search (tea query)

Standalone example matching the working curl template — acquire a token with `swagger-client`, then `POST /search/` against one language dataset.


In [11]:
# ── Example: hybrid search (same payload as the working curl) ──
#
# curl -s -X POST ".../cross-dataset-discovery/search/" \
#   -H "Authorization: Bearer $(curl -s ... client_id=swagger-client ...)" \
#   -d '{"query":"How has the perception of tea changed over time",
#        "k":5, "dataset_ids":["d84d1a2e-..."], "search_mode":"hybrid"}'
#
# Available language corpora (from cross-dataset-discovery/scripts/unify_datasets.py):
LANGUAGE_DATASETS = {
    "encyc":   {"uuid": "07382b91-5bc5-42f9-8391-33adc2460c19", "name": "EncycNet"},
    "kp":      {"uuid": "d84d1a2e-127d-4393-91d0-afb7e4fd9c68", "name": "19th C. Knowledge Project (Britannica)"},
    "diderot": {"uuid": "d5c34990-9acf-4349-91d4-924f58565922", "name": "Encyclopédie (Diderot)"},
    "wiki":    {"uuid": "1f6fba0c-9aea-4345-b5a3-457c924f9e0c", "name": "Wikipedia"},
}

EXAMPLE_DATASET_ID = LANGUAGE_DATASETS["kp"]["uuid"]  # same as curl
EXAMPLE_QUERY = "How has the perception of tea changed over time"

client = CrossDatasetDiscoveryClient(
    base_url=DISCOVERY_URL,
    token=DISCOVERY_TOKEN or None,
)

print(f"Query      : {EXAMPLE_QUERY}")
print(f"Dataset    : {EXAMPLE_DATASET_ID}  ({LANGUAGE_DATASETS['kp']['name']})")
print(f"Mode       : hybrid")
print(f"Token set  : {'yes' if DISCOVERY_TOKEN else 'no'}")
print()

t0 = time.time()
try:
    result = client.search(
        query=EXAMPLE_QUERY,
        k=5,
        dataset_ids=[EXAMPLE_DATASET_ID],
        search_mode="hybrid",
    )
    elapsed = time.time() - t0
    hits = result.get("results", [])
    print(f"✓ {len(hits)} results in {elapsed:.1f}s (server: {result.get('query_time', '?')}ms)\n")
    for j, hit in enumerate(hits, 1):
        snippet = str(hit.get("content", "")).replace("\n", " ")[:160]
        score = hit.get("similarity", "?")
        print(f"{j}. [{score}] {snippet}...")
        print(f"   dataset={hit.get('dataset_id')}  object={hit.get('object_id')}")
        print()
except Exception as exc:
    print(f"✗ Error ({time.time() - t0:.1f}s): {exc}")


Query      : How has the perception of tea changed over time
Dataset    : d84d1a2e-127d-4393-91d0-afb7e4fd9c68  (19th C. Knowledge Project (Britannica))
Mode       : hybrid
Token set  : yes



✓ 5 results in 0.2s (server: 69.72ms)

1. [0.5] In China the hills on which tea is cultivated are not high, and the plantations where the soil is favourable extend over their entire face from the summit to th...
   dataset=d84d1a2e-127d-4393-91d0-afb7e4fd9c68  object=d84d1a2e-127d-4393-91d0-afb7e4fd9c68

2. [0.5] In the meantime, however, tea was not without its apologists. ...
   dataset=d84d1a2e-127d-4393-91d0-afb7e4fd9c68  object=b3f68144-63a2-4486-9a67-0bb2bd320bff

3. [0.33333334] Perception as a psychological term has received various, though related, meanings for different writers. It is sometimes used for the recognition of a sensation...
   dataset=d84d1a2e-127d-4393-91d0-afb7e4fd9c68  object=d84d1a2e-127d-4393-91d0-afb7e4fd9c68

4. [0.33333334] Jonas Hanway (Essay on Tea, 1756) was among its most vigorous assailants. “Men,” he says, “seem to have lost their stature and comeliness, and women their beaut...
   dataset=d84d1a2e-127d-4393-91d0-afb7e4fd9c68  object=d84d1a2e-127d-4

### Retrieve using sub-questions (Q1 Dresden)

Instead of searching with raw `concept + slice_label`, we use the **actual sub-questions from Step 4 (Refine)** as search queries. This matches the real pipeline flow: decompose → retrieve per sub-question.


In [12]:
# ── Retrieve using sub-questions from Step 4 (Refine) ──
# Pick Q1 Dresden — use the Refine sub-questions as search queries.

test_q = QUESTIONS[0]
subs = refine_results.get(test_q, {}).get("subs", [])
if not subs:
    print("No Refine sub-questions found for Q1 — run Step 4c first.")
else:
    print(f"Question: {test_q}")
    print(f"Sub-questions from Refine: {len(subs)}")
    print(f"Search mode: hybrid  |  k=3  |  Token: {'yes' if DISCOVERY_TOKEN else 'no'}")
    print()

    subq_results = {}
    for j, sq in enumerate(subs, 1):
        t0 = time.time()
        try:
            result = client.search(query=sq, k=3, search_mode="hybrid")
            elapsed = time.time() - t0
            hits = result.get("results", [])
            subq_results[j] = {"query": sq, "hits": hits, "time": elapsed}
            print(f"SQ{j} ({elapsed:.1f}s, {len(hits)} hits)")
            print(f"  Q: {sq[:100]}")
            for h_idx, hit in enumerate(hits[:2], 1):
                snippet = str(hit.get("content", "")).replace("\n", " ")[:120]
                print(f"  {h_idx}. [{hit.get('similarity')}] {snippet}...")
        except Exception as exc:
            elapsed = time.time() - t0
            subq_results[j] = {"query": sq, "hits": [], "time": elapsed, "error": str(exc)}
            print(f"SQ{j} ({elapsed:.1f}s) ✗ {exc}")
        print()

    total_hits = sum(len(r["hits"]) for r in subq_results.values())
    print(f"Total: {total_hits} passages across {len(subs)} sub-questions")


Question: How did descriptions of Dresden differ in English sources from 1800 and 1900?
Sub-questions from Refine: 7
Search mode: hybrid  |  k=3  |  Token: yes



SQ1 (0.2s, 3 hits)
  Q: How was Dresden's cultural heritage described in English sources from 1800, and what specific terms 
  1. [0.5] bookId: 25074178-decolonize-your-diet , title: Decolonize Your Diet: Plant-Based Mexican-American Recipes for Health and...
  2. [0.5] Dresden, which is known to have existed in 1206, is of Slavonic origin. ...



SQ2 (0.2s, 3 hits)
  Q: How was Dresden's urban landscape portrayed in English sources from 1800, and what descriptive langu
  1. [0.5] Dresden, which is known to have existed in 1206, is of Slavonic origin. It became the capital of Henry the Illustrious, ...
  2. [0.5] bookId: 3431100-wizard-at-large , title: Wizard at Large , series: The Dresden Files Omnibus #3 , author: Jim Butcher (G...



SQ3 (0.2s, 3 hits)
  Q: What political symbolism was associated with Dresden in English sources from 1800, and how was it fr
  1. [0.5] . Pitt rose to the occasion. He planned a great scheme of union between the two nations (1799). There was to be one parl...
  2. [0.5] Dresden, which is known to have existed in 1206, is of Slavonic origin. ...



SQ4 (0.2s, 3 hits)
  Q: How was Dresden's cultural heritage described in English sources from 1900, and what specific terms 
  1. [0.5] bookId: 25074178-decolonize-your-diet , title: Decolonize Your Diet: Plant-Based Mexican-American Recipes for Health and...
  2. [0.5] Dresden, which is known to have existed in 1206, is of Slavonic origin. ...



SQ5 (0.2s, 3 hits)
  Q: How was Dresden's urban landscape portrayed in English sources from 1900, and what descriptive langu
  1. [0.5] bookId: 10074280-the-dresden-files-collection-1-6 , title: The Dresden Files Collection 1-6 , series: The Dresden Files ...
  2. [0.5] Dresden, which is known to have existed in 1206, is of Slavonic origin. ...



SQ6 (0.2s, 3 hits)
  Q: What political symbolism was associated with Dresden in English sources from 1900, and how was it fr
  1. [0.5] Dresden, which is known to have existed in 1206, is of Slavonic origin. ...
  2. [0.5] DIPLOMACY is the art of conducting the intercourse of nations with each other. The word obviously owes its origin to the...



SQ7 (0.2s, 3 hits)
  Q: How did the frequency of cultural references, tone and sentiment, geopolitical framing, and descript
  1. [0.5] bookId: 752994.The_Misanthrope , title: The Misanthrope , author: Molière , description: The Misanthrope or Le Misanthro...
  2. [0.5] Particulars of his residence at Dresden, more especially of his connexion with the theatre, are given in the memoirs of ...

Total: 21 passages across 7 sub-questions


### Build search queries from pipeline results

For each question that has slices, we build a search query using:
- **concept** from Step 1 (Route) / Step 2 (Disambiguate)
- **slice label** from Step 2
- **slice lemmas** from Step 3 (Extend Knowledge)

The function `build_search_query(concept, slice_label, facets, cross_lingual_tokens)` concatenates these into a single search string.

In [13]:
# ── Build search queries from pipeline results ──
search_queries = {}

for i, q in enumerate(QUESTIONS, 1):
    intent = disamb_results.get(q)
    if intent is None:
        print(f"Q{i}: skipped (no slices)")
        continue

    concept = intent.target_concept
    ext = extend_results.get(q, {})

    queries_for_q = {}
    for s in intent.slices:
        lemmas = ext.get(s.slice_id, [])
        query_str = build_search_query(concept, s.label, [], lemmas)
        queries_for_q[s.slice_id] = {
            "query": query_str,
            "slice": s,
            "lemmas": lemmas,
        }

    search_queries[q] = queries_for_q
    print(f"Q{i}: {concept}")
    for sid, info in queries_for_q.items():
        print(f"  {sid} → \"{info['query']}\"")
    print()

Q1: Dresden
  slice_1 → "Dresden 1800 Dresden Saxon capital Baroque city artistic center"
  slice_2 → "Dresden 1900 Dresden Saxon capital modern city cultural hub"

Q2: marriage
  slice_1 → "marriage German sources, 1850s Ehe Hochzeit Ehepartner Ehevertrag"
  slice_2 → "marriage English sources, 1850s marriage wedding spouse marital contract"

Q3: love
  slice_1 → "love 1820 love adoration devotion fondness"
  slice_2 → "love 1920 love romance affection desire"

Q4: freedom
  slice_1 → "freedom 1830s freedom liberty rights emancipation"
  slice_2 → "freedom 1930s freedom liberty rights autonomy"



### Execute searches per slice

We test all 3 search modes (`sparse`, `dense`, `hybrid`) on one question first, then run hybrid on all questions. Each call sends:

```json
{
  "query": "marriage German sources, 1850s Ehe Hochzeit Ehepartner Ehevertrag",
  "k": 5,
  "search_mode": "hybrid"
}
```

In [14]:
# ── Smoke test: search one question (Q1 Dresden) with all 3 modes ──
# Uses DISCOVERY_TOKEN from the Keycloak cell above (swagger-client).

client = CrossDatasetDiscoveryClient(
    base_url=DISCOVERY_URL,
    token=DISCOVERY_TOKEN or None,
)

# Pick Q1 for the initial smoke test
test_q = QUESTIONS[0]
test_queries = search_queries.get(test_q, {})

if not test_queries:
    print("No queries to test — check that Q1 was disambiguated.")
elif not DISCOVERY_TOKEN:
    print("⚠ No DISCOVERY_TOKEN — re-run the Keycloak auth cell above.")
    print()

if test_queries:
    first_slice_id = list(test_queries.keys())[0]
    test_info = test_queries[first_slice_id]
    test_query_str = test_info["query"]

    print(f"Question: {test_q}")
    print(f"Slice   : {first_slice_id} ({test_info['slice'].label})")
    print(f"Query   : \"{test_query_str}\"")
    print(f"k       : 5")
    print()

    for mode in ["sparse", "dense", "hybrid"]:
        print(f"── {mode.upper()} ──")
        t0 = time.time()
        try:
            result = client.search(
                query=test_query_str,
                k=5,
                search_mode=mode,
            )
            elapsed = time.time() - t0
            hits = result.get("results", [])
            query_time = result.get("query_time", "?")
            print(f"  ✓ {len(hits)} results in {elapsed:.1f}s (server: {query_time}ms)")
            for j, hit in enumerate(hits[:3], 1):
                snippet = str(hit.get("content", ""))[:120]
                score = hit.get("similarity", "?")
                dataset = str(hit.get("dataset_id", "?"))[:8]
                print(f"  {j}. [{score}] {snippet}...")
                print(f"     dataset={dataset}..  obj={str(hit.get('object_id', '?'))[:8]}...")
        except Exception as exc:
            elapsed = time.time() - t0
            print(f"  ✗ Error ({elapsed:.1f}s): {exc}")
        print()


Question: How did descriptions of Dresden differ in English sources from 1800 and 1900?
Slice   : slice_1 (1800)
Query   : "Dresden 1800 Dresden Saxon capital Baroque city artistic center"
k       : 5

── SPARSE ──
  ✓ 5 results in 0.2s (server: 16.7ms)
  1. [65.405876] Louis Marchand, who had reached Dresden on his travels, and lorded it over his artistic colleagues at the Saxon court in...
     dataset=d84d1a2e..  obj=b3f68144...
  2. [62.07115] When Napoleon declared war with Russia in 1812, the Saxon capital was the spot on which the several monarchs of Europe w...
     dataset=d84d1a2e..  obj=d84d1a2e...
  3. [60.17878] MEISSEN , one of the circles into which the kingdom of Saxony is divided, being formed out of the ancient margravate of ...
     dataset=d84d1a2e..  obj=d84d1a2e...

── DENSE ──


  ✓ 5 results in 0.2s (server: 52.63ms)
  1. [0.6059538] DRESDEN , a city of Germany, the capital of the former electorate, now the kingdom of Saxony. It is a beautiful spot, bo...
     dataset=d84d1a2e..  obj=d84d1a2e...
  2. [0.60288787] DRESDEN , a city of Germany, the capital of the former electorate, now the kingdom of Saxony. It is a beautiful spot, bo...
     dataset=d84d1a2e..  obj=b3f68144...
  3. [0.58554924] DRESDEN , the capital of the kingdom of Saxony, is situated in a beautiful and richly cultivated valley on both sides of...
     dataset=d84d1a2e..  obj=d84d1a2e...

── HYBRID ──


  ✓ 5 results in 0.2s (server: 47.78ms)
  1. [0.5] Louis Marchand, who had reached Dresden on his travels, and lorded it over his artistic colleagues at the Saxon court in...
     dataset=d84d1a2e..  obj=b3f68144...
  2. [0.5] DRESDEN , a city of Germany, the capital of the former electorate, now the kingdom of Saxony. It is a beautiful spot, bo...
     dataset=d84d1a2e..  obj=d84d1a2e...
  3. [0.41025642] DRESDEN , a city of Germany, the capital of the former electorate, now the kingdom of Saxony. It is a beautiful spot, bo...
     dataset=d84d1a2e..  obj=b3f68144...



### Full retrieval: all questions, all slices (hybrid mode)

Run one search per slice for every question, using the `CrossDatasetMultiSliceRetriever` from the pipeline.

In [15]:
# ── Full retrieval: all questions × all slices (hybrid) ──
from src.retrieval.multi_slice_retriever import CrossDatasetMultiSliceRetriever

retriever = CrossDatasetMultiSliceRetriever(client=client, k=5)
retrieval_results = {}

for i, q in enumerate(QUESTIONS, 1):
    intent = disamb_results.get(q)
    if intent is None:
        print(f"Q{i}: skipped")
        continue

    concept = intent.target_concept
    ext = extend_results.get(q, {})
    variants_by_slice = {
        s.slice_id: ext.get(s.slice_id, []) for s in intent.slices
    }

    t0 = time.time()
    try:
        passages, errors, queries = retriever.retrieve_all(
            concept, intent.slices, variants_by_slice
        )
        elapsed = time.time() - t0

        retrieval_results[q] = {
            "passages": passages,
            "errors": errors,
            "queries": queries,
        }

        total = sum(len(p) for p in passages.values())
        print(f"Q{i}  ({elapsed:.1f}s)  {total} passages across {len(passages)} slices")
        for sid in sorted(passages.keys()):
            n = len(passages[sid])
            err = errors.get(sid, "")
            qry = queries.get(sid, "")
            print(f"  {sid}: {n} results  query=\"{qry[:80]}\"")
            if err:
                print(f"    ✗ {err}")
            for j, p in enumerate(passages[sid][:2], 1):
                print(f"    {j}. [{p.doc_id[:12]}...] {p.text[:100]}...")
    except Exception as exc:
        elapsed = time.time() - t0
        print(f"Q{i}  ✗ Error ({elapsed:.1f}s): {exc}")
    print()

Q1  (0.3s)  10 passages across 2 slices
  slice_1: 5 results  query="Dresden 1800 Dresden Saxon capital Baroque city artistic center"
    1. [d84d1a2e-127...] DRESDEN , a city of Germany, the capital of the former electorate, now the kingdom of Saxony. It is ...
    2. [b3f68144-63a...] Louis Marchand, who had reached Dresden on his travels, and lorded it over his artistic colleagues a...
  slice_2: 5 results  query="Dresden 1900 Dresden Saxon capital modern city cultural hub"
    1. [d84d1a2e-127...] The preponderating industrial activity of Saxony fosters the tendency of the population to concentra...
    2. [d84d1a2e-127...] When Napoleon declared war with Russia in 1812, the Saxon capital was the spot on which the several ...



Q2  (4.3s)  10 passages across 2 slices
  slice_1: 5 results  query="marriage German sources, 1850s Ehe Hochzeit Ehepartner Ehevertrag"
    1. [d84d1a2e-127...] Middle German language, 518....
    2. [b3f68144-63a...] Numerous vase-paintings, representing a bridal procession, which Forster {Ζcτιs und Hera's Hochzeit)...
  slice_2: 5 results  query="marriage English sources, 1850s marriage wedding spouse marital contract"
    1. [d84d1a2e-127...] DIVORCE is the dissolution of the relationship of marriage. Few social questions are surrounded with...
    2. [b3f68144-63a...]  as well as from the manner in which marriages are contracted. ...



Q3  (5.2s)  10 passages across 2 slices
  slice_1: 5 results  query="love 1820 love adoration devotion fondness"
    1. [d84d1a2e-127...] 1820 „...
    2. [b3f68144-63a...] The lady was very handsome, childless, fond of music, twenty-three; she proved to him a loving and d...
  slice_2: 5 results  query="love 1920 love romance affection desire"
    1. [b3f68144-63a...]  the Histoire Amoureuse. ...
    2. [d84d1a2e-127...] . The threefold division into passions and affections, self-love and benevolence, and conscience, is...



Q4  (4.4s)  10 passages across 2 slices
  slice_1: 5 results  query="freedom 1830s freedom liberty rights emancipation"
    1. [b3f68144-63a...] Though a strict adherent of the creed of Rome, he was a Liberal, nay a Radical, as regards measures ...
    2. [d84d1a2e-127...] 1830...
  slice_2: 5 results  query="freedom 1930s freedom liberty rights autonomy"
    1. [b3f68144-63a...] sided freedom which date from the latter half of the 18th century. ...
    2. [d84d1a2e-127...] BOROUGH . Although the idea of self-government by a town is exemplified in the colonice and municipi...



### Results summary

Overview of what was retrieved per question and slice.

In [16]:
# ── Summary table ──
rows = []
for i, q in enumerate(QUESTIONS, 1):
    rr = retrieval_results.get(q)
    if rr is None:
        rows.append(
            f"<tr><td style='padding:6px 10px'>Q{i}</td>"
            f"<td style='padding:6px 10px'>{q[:50]}...</td>"
            f"<td colspan=4 style='padding:6px 10px;color:#888'>skipped</td></tr>"
        )
        continue
    passages = rr["passages"]
    errors = rr["errors"]
    queries = rr["queries"]
    for sid in sorted(passages.keys()):
        n = len(passages[sid])
        err = errors.get(sid, "")
        qry = queries.get(sid, "")
        status = f"✗ {err}" if err else f"✓ {n} passages"
        # Show first passage snippet
        snippet = passages[sid][0].text[:80] + "..." if passages[sid] else "—"
        rows.append(
            f"<tr><td style='padding:6px 10px'>Q{i}</td>"
            f"<td style='padding:6px 10px'>{sid}</td>"
            f"<td style='padding:6px 10px'>{status}</td>"
            f"<td style='padding:6px 10px;font-size:12px'>{qry[:60]}...</td>"
            f"<td style='padding:6px 10px;font-size:11px;color:#666'>{snippet}</td></tr>"
        )

html = (
    '<table style="border-collapse:collapse;margin:10px 0;font-size:14px">'
    '<tr style="background:#f0f0f0">'
    "<th style='padding:6px 10px'>Q</th>"
    "<th style='padding:6px 10px'>Slice</th>"
    "<th style='padding:6px 10px'>Status</th>"
    "<th style='padding:6px 10px'>Search Query</th>"
    "<th style='padding:6px 10px'>First Passage</th></tr>"
    + "\n".join(rows)
    + "</table>"
)
display(HTML(html))

Q,Slice,Status,Search Query,First Passage
Q1,slice_1,✓ 5 passages,Dresden 1800 Dresden Saxon capital Baroque city artistic cen...,"DRESDEN , a city of Germany, the capital of the former electorate, now the kingd..."
Q1,slice_2,✓ 5 passages,Dresden 1900 Dresden Saxon capital modern city cultural hub...,The preponderating industrial activity of Saxony fosters the tendency of the pop...
Q2,slice_1,✓ 5 passages,"marriage German sources, 1850s Ehe Hochzeit Ehepartner Eheve...","Middle German language, 518...."
Q2,slice_2,✓ 5 passages,"marriage English sources, 1850s marriage wedding spouse mari...",DIVORCE is the dissolution of the relationship of marriage. Few social questions...
Q3,slice_1,✓ 5 passages,love 1820 love adoration devotion fondness...,1820 „...
Q3,slice_2,✓ 5 passages,love 1920 love romance affection desire...,the Histoire Amoureuse. ...
Q4,slice_1,✓ 5 passages,freedom 1830s freedom liberty rights emancipation...,"Though a strict adherent of the creed of Rome, he was a Liberal, nay a Radical, ..."
Q4,slice_2,✓ 5 passages,freedom 1930s freedom liberty rights autonomy...,sided freedom which date from the latter half of the 18th century. ...


---
## Live ThematicExploration API (traceable JSON)

Each pipeline step is listed with its outcome, then the full JSON (`pipeline[].output`).


In [ ]:
import html
import json
import os
import urllib.request
from IPython.display import HTML, JSON, Markdown, display

API = os.getenv("LANGUAGE_PILOT_URL", "https://datagems-dev.scayle.es/language-pilot").rstrip("/")
QUESTION = QUESTIONS[1] if "QUESTIONS" in globals() else (
    "How did a marriage look like in the 1800s compared to now?"
)

payload = json.dumps({"query": QUESTION, "query_id": "nb-pretty", "k": 3}).encode()
request = urllib.request.Request(
    f"{API}/ThematicExploration",
    data=payload,
    headers={"Content-Type": "application/json", "Accept": "application/json"},
    method="POST",
)
with urllib.request.urlopen(request, timeout=180) as http:
    result = json.loads(http.read().decode())

steps = result.get("pipeline") or []
rows = []
for item in steps:
    color = {"ok": "#0f6e56", "partial": "#b45309", "halted": "#b91c1c", "failed": "#b91c1c"}.get(
        item.get("outcome"), "#334155"
    )
    rows.append(
        "<tr>"
        f"<td style='padding:8px 10px'>{item.get('step')}</td>"
        f"<td style='padding:8px 10px'><b>{html.escape(str(item.get('title') or item.get('name')))}</b></td>"
        f"<td style='padding:8px 10px;color:{color};font-weight:600'>{html.escape(str(item.get('outcome')))}</td>"
        f"<td style='padding:8px 10px'>{html.escape(str(item.get('summary') or ''))}</td>"
        "</tr>"
    )

display(Markdown(f"### `{API}/ThematicExploration` — pipeline trace"))
if rows:
    display(HTML(
        "<table style='border-collapse:collapse;width:100%;font:14px/1.4 system-ui'>"
        "<tr style='background:#f1f5f9;text-align:left'>"
        "<th style='padding:8px 10px'>#</th><th style='padding:8px 10px'>Step</th>"
        "<th style='padding:8px 10px'>Outcome</th><th style='padding:8px 10px'>What happened</th></tr>"
        + "".join(rows) + "</table>"
    ))
else:
    display(Markdown("_This cluster image does not yet return `pipeline`. Redeploy v0.1.2 to get the step trace._"))

pretty = {
    "question": result.get("question"),
    "status": result.get("status"),
    "synthesized_answer": result.get("synthesized_answer"),
    "grounded_citations": result.get("grounded_citations"),
    "pipeline": steps or None,
}
pretty = {k: v for k, v in pretty.items() if v not in (None, [], {})}
display(Markdown("### Traceable JSON"))
try:
    display(JSON(pretty, expanded=False))
except Exception:
    pass
text = json.dumps(pretty, indent=2, ensure_ascii=False, default=str)
display(HTML(
    "<pre style='background:#0f172a;color:#e2e8f0;padding:16px 18px;"
    "border-radius:12px;overflow:auto;max-height:720px;"
    "font:13px/1.45 ui-monospace,SFMono-Regular,Menlo,monospace;"
    "box-shadow:inset 0 0 0 1px #1e293b'>"
    + html.escape(text) + "</pre>"
))
